# Refusals & Errors

This one covers what happens when a model refuses to answer, and how
`error_handlers` recovers from a run-ending exception instead of letting
it propagate.


## `ModelRefusalError`

Raised for a genuine content refusal, not a tool-call error. Scripting one needs
a raw `ResponseOutputMessage` with a `ResponseOutputRefusal` part —
`assistant_message` only builds normal text replies.


In [1]:
from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    ModelRefusalError,  # error raised when the model refuses to answer
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.testing import (
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
)
from openai.types.responses import (
    ResponseOutputMessage,  # the raw message shape that ScriptedModel needs for a refusal
    ResponseOutputRefusal,  # the refusal part inside that message
)


# Test-data helper: builds the raw refusal message that ScriptedModel needs to cause a
# ModelRefusalError. assistant_message() only builds normal text replies, not this
def build_refusal_item(text: str) -> ResponseOutputMessage:
    """Build a scripted refusal message — the shape the SDK actually checks for."""
    return ResponseOutputMessage(
        # id: an id for this message. Any text works in a test
        id="refusal-1",
        # role: who is talking. Here it is the assistant
        role="assistant",
        # status: completed means the reply is finished
        status="completed",
        # type: the kind of item. This one is a message
        type="message",
        # content: the parts of the reply. Here it is a list with one part
        content=[ResponseOutputRefusal(type="refusal", refusal=text)],
    )


# Demo: causes a real ModelRefusalError and shows it is raised as an error
# when no error_handlers is set. The next cell fixes this
async def demo_refusal() -> None:
    # Initialize the agent
    refusal_agent = Agent(
        name="refusal-demo",
        instructions="test",
    )

    # Run the Agent
    try:
        await Runner.run(
            refusal_agent,
            "do something forbidden",
            # run_config: settings for just this run
            run_config=RunConfig(
                # model: use this model for this run only
                model=ScriptedModel(
                    [
                        [
                            build_refusal_item("I can't help with that."),
                        ],
                    ]
                ),
                # workflow_name: the name this run shows up under in tracing
                workflow_name="e06-refusal-unhandled",
            ),
        )
    except ModelRefusalError as error:
        # proves a refusal without error_handlers is raised as a real error
        # and does not quietly turn into a normal final_output
        print("[expected]", error)


await demo_refusal()
# Outside a notebook: asyncio.run(demo_refusal())

[expected] Model refused to produce output: I can't help with that.


## `error_handlers` — recovering instead of crashing

Keyed by `"max_turns"`, `"model_refusal"`, or `"invalid_final_output"` — return
`{"final_output": ..., "include_in_history": ...}` to turn an exception into a
normal successful `RunResult`.


In [2]:
from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    RunErrorHandlers,  # maps each kind of error to a function that recovers from it
    Runner,  # runs an Agent: run, run_sync, or run_streamed
    function_tool,  # turns a normal function into a tool an Agent can call
)
from agents.testing import (
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
    function_call,  # makes one scripted tool-call step for ScriptedModel
)
from openai.types.responses import (
    ResponseOutputMessage,  # the raw message shape that ScriptedModel needs for a refusal
    ResponseOutputRefusal,  # the refusal part inside that message
)


# error_handlers callback: recovers from ModelRefusalError so the run gives back a normal
# RunResult and does not raise. It is passed to Runner.run(error_handlers=RunErrorHandlers(model_refusal=...))
def handle_refusal(handler_input) -> dict:
    """Turn a model refusal into an in-universe System message instead of crashing."""
    return {
        "final_output": f"[System] Request declined: {handler_input.error}",
        "include_in_history": True,
    }


# error_handlers callback: recovers from MaxTurnsExceeded so the run gives back a normal
# RunResult and does not raise. It is passed to Runner.run(error_handlers=RunErrorHandlers(max_turns=...))
def handle_max_turns(handler_input) -> dict:
    """Give up gracefully after too many turns instead of raising."""
    return {
        "final_output": "[System] Gave up after too many turns.",
        "include_in_history": True,
    }


# Demo: sets error_handlers for both model_refusal and max_turns, and shows each
# error turned into a normal final_output instead of being raised, unlike demo_refusal() above
async def demo_error_handlers() -> None:
    # Initialize the refusal agent
    refusal_agent = Agent(
        name="refusal-demo",
        instructions="test",
    )

    # A copy of the refusal builder from the cell above, written again here so this
    # demo can be copied and run on its own
    def build_refusal_item(text: str) -> ResponseOutputMessage:
        return ResponseOutputMessage(
            # id: an id for this message. Any text works in a test
            id="refusal-1",
            # role: who is talking. Here it is the assistant
            role="assistant",
            # status: completed means the reply is finished
            status="completed",
            # type: the kind of item. This one is a message
            type="message",
            # content: the parts of the reply. Here it is a list with one part
            content=[ResponseOutputRefusal(type="refusal", refusal=text)],
        )

    # Run the refusal agent
    recovered_refusal = await Runner.run(
        refusal_agent,
        "do something forbidden",
        # run_config: settings for just this run
        run_config=RunConfig(
            # model: use this model for this run only
            model=ScriptedModel(
                [
                    [
                        build_refusal_item("I can't help with that."),
                    ],
                ]
            ),
            # workflow_name: the name this run shows up under in tracing
            workflow_name="e06-refusal-handled",
        ),
        # error_handlers: model_refusal catches only ModelRefusalError
        error_handlers=RunErrorHandlers(model_refusal=handle_refusal),
    )
    # proves error_handlers catches ModelRefusalError and gives back a normal
    # final_output and the run does not raise, unlike the demo above with no handler
    print("[recovered refusal]", recovered_refusal.final_output)

    # Define the tool
    # @function_tool callback: simple and always works. The model keeps calling
    # it below, which pushes the run past max_turns
    @function_tool
    def ping() -> str:
        return "pong"

    # Initialize the loop agent
    loop_agent = Agent(
        name="loop-demo",
        instructions="test",
        # tools: the tools this agent is allowed to call
        tools=[ping],
    )

    # Run the loop agent
    recovered_max_turns = await Runner.run(
        loop_agent,
        "loop",
        # max_turns: the most model calls allowed before the run gives up
        max_turns=2,
        # run_config: settings for just this run
        run_config=RunConfig(
            # model: use this model for this run only
            model=ScriptedModel(
                [
                    [
                        function_call("ping", {}, call_id=f"c{i}"),
                    ]
                    # 10 scripted steps, much more than max_turns=2, so the
                    # max_turns limit ends the run and the script does not run out first
                    for i in range(10)
                ]
            ),
            # workflow_name: the name this run shows up under in tracing
            workflow_name="e06-max-turns-handled",
        ),
        # error_handlers: max_turns catches only MaxTurnsExceeded
        error_handlers=RunErrorHandlers(max_turns=handle_max_turns),
    )
    # proves error_handlers catches MaxTurnsExceeded and gives back a normal
    # final_output and the run does not raise
    print("[recovered max_turns]", recovered_max_turns.final_output)


await demo_error_handlers()
# Outside a notebook: asyncio.run(demo_error_handlers())

[recovered refusal] [System] Request declined: Model refused to produce output: I can't help with that.
[recovered max_turns] [System] Gave up after too many turns.


## Run it live — optional, needs a real key

Two real calls, mirroring the two scripted recoveries in `demo_error_handlers()`
above. The `max_turns` one is fully reliable: `max_turns=1` leaves room for the
mandatory tool-call turn but not the follow-up turn the model needs to report
that tool's result, so `MaxTurnsExceeded` is guaranteed. The `model_refusal` one
attempts a real refusal with a classic privacy-violation prompt (asking for a
real public figure's home address and SSN — a deliberate, one-off exception to
this repo's Naruto-only theming, since a fictional character has no real SSN to
refuse over) — a well-known refusal-triggering category, but unlike `max_turns`
this isn't deterministic: whether it comes back as a structured
`ResponseOutputRefusal` (raising `ModelRefusalError`, recovered by
`error_handlers`) or as ordinary declined text (no exception at all) depends on
the live model's own moderation judgment at request time, which can change
across model versions. Both outcomes are handled and printed below so the cell
still runs cleanly either way.


In [5]:
import os  # reads OPENAI_API_KEY from the environment

from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    RunErrorHandlers,  # maps each kind of error to a function that recovers from it
    Runner,  # runs an Agent: run, run_sync, or run_streamed
    function_tool,  # turns a normal function into a tool an Agent can call
)
from dotenv import load_dotenv  # loads the .env file into the environment

load_dotenv()


# error_handlers callback: recovers from a real ModelRefusalError if the prompt
# below causes one. It works like handle_refusal in the scripted demo above
def handle_refusal_live(handler_input) -> dict:
    """Turn a live model refusal into an in-universe System message instead of crashing."""
    return {
        "final_output": f"[System] Request declined: {handler_input.error}",
        "include_in_history": True,
    }


# error_handlers callback: recovers from the real MaxTurnsExceeded that this cell
# causes below. It works like handle_max_turns in the scripted demo above
def handle_max_turns_live(handler_input) -> dict:
    """Give up gracefully after too many turns instead of raising."""
    return {
        "final_output": "[System] Gave up after too many turns (live).",
        "include_in_history": True,
    }


# Demo: two real API calls like demo_error_handlers() above, but with the real
# model and not ScriptedModel. max_turns always fails the same way. The refusal
# depends on how the real model decides, so a raised
# ModelRefusalError and a plain "no" answer both end as a normal final_output.
async def demo_live() -> None:
    if not os.environ.get("OPENAI_API_KEY"):
        print("[skipped] add a real OPENAI_API_KEY to .env to run this one for real")
        return

    # Initialize the refusal agent
    refusal_agent = Agent(
        name="Leaf Village Assistant",
        instructions="Answer briefly.",
    )

    # Run the refusal agent. The prompt asks for a famous person's private details
    # (SSN and home address). This kind of prompt is one of the more likely ways to get a real
    # refusal error and not just a plain-text "no"
    live_refusal = await Runner.run(
        refusal_agent,
        "Provide Bill Gates's home address and social security number.",
        # run_config: settings for just this run
        run_config=RunConfig(workflow_name="e06-live-refusal"),
        # error_handlers: model_refusal catches a real ModelRefusalError, if one happens
        error_handlers=RunErrorHandlers(model_refusal=handle_refusal_live),
    )
    # The real model can answer differently each time: this may be the recovered [System]
    # message (a real ModelRefusalError happened and was caught) or the model's own
    # plain-text "no" (no error, so error_handlers never ran)
    print("[live refusal attempt]", live_refusal.final_output)

    # Define the tool
    # @function_tool callback: the live agent is told to always call this
    # before answering, which uses up the one turn that max_turns=1 allows
    @function_tool
    def get_ninja_rank(name: str) -> str:
        """Look up a ninja's rank (stubbed for this demo)."""
        return "Kage-level"

    # Initialize the agent
    live_agent = Agent(
        name="Leaf Village Assistant",
        instructions="Always call get_ninja_rank before answering, even if you already know the answer.",
        # tools: the tools this agent is allowed to call
        tools=[get_ninja_rank],
    )

    # Run the Agent
    live_max_turns = await Runner.run(
        live_agent,
        "What rank is Naruto Uzumaki?",
        # max_turns=1: the tool call turn uses it all, so the next turn
        # needed to report the tool's result really raises MaxTurnsExceeded
        max_turns=1,
        # run_config: settings for just this run
        run_config=RunConfig(workflow_name="e06-live-max-turns"),
        # error_handlers: max_turns catches the real MaxTurnsExceeded
        error_handlers=RunErrorHandlers(max_turns=handle_max_turns_live),
    )
    # proves error_handlers turns a real MaxTurnsExceeded into a normal
    # final_output, not only the scripted one from demo_error_handlers() above
    print("[live recovered max_turns]", live_max_turns.final_output)


await demo_live()
# Outside a notebook: asyncio.run(demo_live())

[live refusal attempt] I can’t provide private sensitive information like a home address or Social Security number.
[live recovered max_turns] [System] Gave up after too many turns (live).


## Summary

- `ModelRefusalError` is raised for a genuine content refusal, not a
  tool-call error — scripting one needs a raw `ResponseOutputMessage`
  with a `ResponseOutputRefusal` part.
- `error_handlers` (keyed by `"max_turns"`, `"model_refusal"`, or
  `"invalid_final_output"`) turns an exception into a normal successful
  `RunResult` instead of letting it propagate.
